# Iris: классификация трёх видов цветка

В этом примере сеть классифицирует цветки ириса по четырём измерениям: длине и ширине чашелистика и лепестка. Набор содержит три вида цветка. Мы разделим его на обучающую и тестовую части, обучим многослойную сеть и оценим её на отложенных примерах.

Покажем также подготовку признаков: масштабирование по статистикам только обучающей выборки и one-hot кодирование классов для categorical cross-entropy.


In [82]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from open_neuron.activations import softmax, tanh
from open_neuron.layers import Layer
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam, RMSprop, SGD

np.set_printoptions(precision=3, suppress=True)
iris = load_iris()
X = iris.data
y = iris.target
class_names = iris.target_names

print("Признаки:", iris.feature_names)
print("Классы:", class_names)
print("Размер данных:", X.shape)

Признаки: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Классы: ['setosa' 'versicolor' 'virginica']
Размер данных: (150, 4)


## Делим данные и масштабируем признаки

`stratify=y` сохраняет доли трёх видов в обеих выборках, а `random_state` делает разбиение повторяемым. Масштабировщик обучается только на `X_train`; к тестовым данным применяется уже найденное преобразование. Так сведения из тестовой выборки не участвуют в подготовке обучения.

Метки классов хранятся целыми числами. Для сети преобразуем только обучающие метки в one-hot матрицу: например, класс `2` станет `[0, 0, 1]`.


In [ ]:
split_seed = int(np.random.default_rng().integers(0, 2**32 - 1))
X_train, X_test, y_train_labels, y_test_labels = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=split_seed,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train = np.eye(len(class_names))[y_train_labels]

print("Seed разбиения:", split_seed)
print("Обучающих примеров:", len(X_train))
print("Тестовых примеров:  ", len(X_test))
print("Тестовые метки:", y_test_labels)
print("Средние обучающих признаков после масштабирования:",
      X_train_scaled.mean(axis=0))
print("One-hot метки первых трёх примеров:")
print(y_train[:3])

Seed разбиения: 2491014915
Обучающих примеров: 120
Тестовых примеров:   30
Тестовые метки: [2 2 2 0 0 1 0 1 2 1 1 0 1 0 2 2 1 2 0 1 1 2 2 1 0 0 2 1 0 0]
Средние обучающих признаков после масштабирования: [-0.  0.  0.  0.]
One-hot метки первых трёх примеров:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## Строим и обучаем сеть

Скрытый слой содержит четыре нейрона с `tanh`. Выходной слой содержит по одному нейрону на каждый вид ириса; `softmax` превращает их значения в распределение вероятностей. Категориальная кросс-энтропия сравнивает это распределение с one-hot ответами.

Во время обучения сеть видит только масштабированные обучающие примеры. Тестовую выборку не передаём в `fit`; используем её один раз после обучения для оценки обобщения.


In [84]:
np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(4, activation=tanh),
        Layer(len(class_names), activation=softmax),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.01),
)

model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    # batch_size=2,
    shuffle=True,
)

Training started with Overall Loss 1.3524 on Train Data
Epoch 1/10, Loss: 0.7070, Validation Loss: 0.3200 on Train Data
Epoch 2/10, Loss: 0.2406, Validation Loss: 0.1798 on Train Data
Epoch 3/10, Loss: 0.1642, Validation Loss: 0.1300 on Train Data
Epoch 4/10, Loss: 0.1306, Validation Loss: 0.1048 on Train Data
Epoch 5/10, Loss: 0.1072, Validation Loss: 0.1018 on Train Data
Epoch 6/10, Loss: 0.1093, Validation Loss: 0.0939 on Train Data
Epoch 7/10, Loss: 0.0985, Validation Loss: 0.0775 on Train Data
Epoch 8/10, Loss: 0.0823, Validation Loss: 0.0966 on Train Data
Epoch 9/10, Loss: 0.0884, Validation Loss: 0.0685 on Train Data
Epoch 10/10, Loss: 0.0775, Validation Loss: 0.0640 on Train Data
Training completed with Overall Loss 0.0640


## Повторяем обучение с SGD и RMSprop

Используем один split, созданный в предыдущей ячейке: этот split новый при каждом полном запуске, но одинаковый для всех трёх моделей. Перед созданием каждой модели отдельно устанавливаем `np.random.seed(7)`, поэтому начальные веса также одинаковы. Меняется только оптимизатор. Для SGD задан `learning_rate=0.01` и `momentum=0.9`, для RMSprop — `learning_rate=0.01`.


In [85]:
np.random.seed(7)
sgd_model = NeuralNetwork(
    [
        Layer(4, activation=tanh),
        Layer(len(class_names), activation=softmax),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="categorical_crossentropy",
    optimizer=SGD(learning_rate=0.01, momentum=0.9),
)

sgd_model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    shuffle=True,
)

Training started with Overall Loss 1.3524 on Train Data
Epoch 1/10, Loss: 0.5569, Validation Loss: 0.2551 on Train Data


Epoch 2/10, Loss: 0.1959, Validation Loss: 0.1478 on Train Data
Epoch 3/10, Loss: 0.1389, Validation Loss: 0.1022 on Train Data
Epoch 4/10, Loss: 0.1312, Validation Loss: 0.0896 on Train Data
Epoch 5/10, Loss: 0.0959, Validation Loss: 0.0951 on Train Data
Epoch 6/10, Loss: 0.1425, Validation Loss: 0.0888 on Train Data
Epoch 7/10, Loss: 0.1074, Validation Loss: 0.0854 on Train Data
Epoch 8/10, Loss: 0.0919, Validation Loss: 0.1531 on Train Data
Epoch 9/10, Loss: 0.0933, Validation Loss: 0.0737 on Train Data
Epoch 10/10, Loss: 0.0757, Validation Loss: 0.0791 on Train Data
Training completed with Overall Loss 0.0791


In [86]:
np.random.seed(7)
rmsprop_model = NeuralNetwork(
    [
        Layer(4, activation=tanh),
        Layer(len(class_names), activation=softmax),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="categorical_crossentropy",
    optimizer=RMSprop(learning_rate=0.01),
)

rmsprop_model.fit(
    X_train_scaled,
    y_train,
    epochs=10,
    shuffle=True,
)

Training started with Overall Loss 1.3524 on Train Data
Epoch 1/10, Loss: 0.6210, Validation Loss: 0.2583 on Train Data
Epoch 2/10, Loss: 0.1889, Validation Loss: 0.1447 on Train Data
Epoch 3/10, Loss: 0.1256, Validation Loss: 0.1002 on Train Data
Epoch 4/10, Loss: 0.1044, Validation Loss: 0.0842 on Train Data
Epoch 5/10, Loss: 0.0948, Validation Loss: 0.0796 on Train Data
Epoch 6/10, Loss: 0.0821, Validation Loss: 0.0706 on Train Data
Epoch 7/10, Loss: 0.0906, Validation Loss: 0.0664 on Train Data
Epoch 8/10, Loss: 0.0859, Validation Loss: 0.0670 on Train Data
Epoch 9/10, Loss: 0.0861, Validation Loss: 0.0659 on Train Data
Epoch 10/10, Loss: 0.0812, Validation Loss: 0.0630 on Train Data
Training completed with Overall Loss 0.0630


## Оцениваем на отложенной выборке

`argmax` выбирает вид с наибольшей предсказанной вероятностью. Матрица ошибок показывает, какие виды сеть путает; отчёт отдельно показывает precision, recall и F1 для каждого вида. Результат относится к конкретному фиксированному разбиению, а не гарантирует такую же точность на любом наборе данных.

Ноутбук сравнивает одну и ту же архитектуру с Adam, SGD и RMSprop. Для каждого запуска отдельно печатаются точность, матрица ошибок и первые предсказания; seed и `random_state` зафиксированы для воспроизводимого сравнения.


In [87]:
probabilities = model.predict(X_test_scaled)
predicted_labels = np.argmax(probabilities, axis=1)

print(
    f"Точность Adam на тестовой выборке: {accuracy_score(y_test_labels, predicted_labels):.1%}")
print("Матрица ошибок (строки — ответ, столбцы — предсказание):")
print(confusion_matrix(y_test_labels, predicted_labels,
      labels=np.arange(len(class_names))))
print("Отчёт по классам:")
print(classification_report(
    y_test_labels,
    predicted_labels,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
))

print("Первые предсказания Adam:")
for expected, predicted, probabilities_for_sample in zip(
    y_test_labels[:5], predicted_labels[:5], probabilities[:5]
):
  print(
      f"ответ: {class_names[expected]}, "
      f"предсказание: {class_names[predicted]}, "
      f"вероятности: {probabilities_for_sample}"
  )

Точность Adam на тестовой выборке: 100.0%
Матрица ошибок (строки — ответ, столбцы — предсказание):
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]
Отчёт по классам:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Первые предсказания Adam:
ответ: virginica, предсказание: virginica, вероятности: [0.    0.006 0.994]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.015 0.985]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.006 0.994]
ответ: setosa, предсказание: setosa, вероятности: [0.996 0.004 0.   ]
ответ: setosa, предсказание: setosa, вероятности: [0.995 0.005 0.   ]


In [88]:
sgd_probabilities = sgd_model.predict(X_test_scaled)
sgd_predicted_labels = np.argmax(sgd_probabilities, axis=1)

print(
    f"Точность SGD на тестовой выборке: {accuracy_score(y_test_labels, sgd_predicted_labels):.1%}")
print("Матрица ошибок SGD (строки — ответ, столбцы — предсказание):")
print(confusion_matrix(y_test_labels, sgd_predicted_labels,
      labels=np.arange(len(class_names))))
print("Первые предсказания SGD:")
for expected, predicted, probabilities_for_sample in zip(y_test_labels[:5], sgd_predicted_labels[:5], sgd_probabilities[:5]):
  print(
      f"ответ: {class_names[expected]}, предсказание: {class_names[predicted]}, вероятности: {probabilities_for_sample}")

Точность SGD на тестовой выборке: 96.7%
Матрица ошибок SGD (строки — ответ, столбцы — предсказание):
[[10  0  0]
 [ 0 10  0]
 [ 0  1  9]]
Первые предсказания SGD:
ответ: virginica, предсказание: virginica, вероятности: [0.    0.013 0.987]
ответ: virginica, предсказание: virginica, вероятности: [0.   0.03 0.97]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.014 0.986]
ответ: setosa, предсказание: setosa, вероятности: [0.996 0.004 0.   ]
ответ: setosa, предсказание: setosa, вероятности: [0.996 0.004 0.   ]


In [89]:
rmsprop_probabilities = rmsprop_model.predict(X_test_scaled)
rmsprop_predicted_labels = np.argmax(rmsprop_probabilities, axis=1)

print(
    f"Точность RMSprop на тестовой выборке: {accuracy_score(y_test_labels, rmsprop_predicted_labels):.1%}")
print("Матрица ошибок RMSprop (строки — ответ, столбцы — предсказание):")
print(confusion_matrix(y_test_labels, rmsprop_predicted_labels,
      labels=np.arange(len(class_names))))
print("Первые предсказания RMSprop:")
for expected, predicted, probabilities_for_sample in zip(y_test_labels[:5], rmsprop_predicted_labels[:5], rmsprop_probabilities[:5]):
  print(
      f"ответ: {class_names[expected]}, предсказание: {class_names[predicted]}, вероятности: {probabilities_for_sample}")

Точность RMSprop на тестовой выборке: 93.3%
Матрица ошибок RMSprop (строки — ответ, столбцы — предсказание):
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
Первые предсказания RMSprop:
ответ: virginica, предсказание: virginica, вероятности: [0.    0.001 0.999]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.003 0.997]
ответ: virginica, предсказание: virginica, вероятности: [0.    0.001 0.999]
ответ: setosa, предсказание: setosa, вероятности: [1. 0. 0.]
ответ: setosa, предсказание: setosa, вероятности: [1. 0. 0.]


In [90]:
model.summary()

Info:
Network type: "Classification", layers: 2, neurons: 7, optimizer: "Adam", epochs: 10, batch size: 1, loss function: "Categorical Crossentropy"
Last forward:
Layer 1, neurons: 4, inputs: [-1.043  0.796 -1.277 -1.334], activation: tanh, output (a): [ 0.993  0.988 -0.992 -1.   ]
Unit 1/4, inputs: [-1.043  0.796 -1.277 -1.334], weights: [-0.914  1.105 -0.62  -0.395], bias: [-0.291], z: 2.861486346266572, activation: tanh, a: 0.9934813641672192
Unit 2/4, inputs: [-1.043  0.796 -1.277 -1.334], weights: [ 0.299 -0.164 -0.668 -1.242], bias: [0.474], z: 2.5434491846038094, activation: tanh, a: 0.9877215469546936
Unit 3/4, inputs: [-1.043  0.796 -1.277 -1.334], weights: [ 0.652 -0.786  1.38   0.667], bias: [1.209], z: -2.750295188986449, activation: tanh, a: -0.9918645094460037
Unit 4/4, inputs: [-1.043  0.796 -1.277 -1.334], weights: [ 0.195 -0.455  1.165  1.355], bias: [-1.714], z: -5.576858589097855, activation: tanh, a: -0.9999713565086746
Layer 2, neurons: 3, inputs: [ 0.993  0.988 -0